# 🤖 Model Documentation — Pump Failure Prediction MVP

## 1. Experiment Registry

All experiments tracked via MLflow.
Tracking URI: `sqlite:///mlflow.db`
Experiment name: `pump-failure-prediction`

### 1.1 Runs Summary

| Run | Model | F1-Macro (CV) | F1-Macro (Val) | ROC-AUC (Val) | Status |
|---|---|---|---|---|---|
| Optuna_Tuning | XGBoost (50 trials) | 0.9990 | — | — | Tuning |
| XGBoost_Tuned | XGBoost | 0.9992 | **0.9972** | **1.0000** | ✅ Production |
| RandomForest | Random Forest | — | baseline | baseline | Baseline |

### 1.2 Optuna Best Hyperparameters

| Hyperparameter | Search Range | Best Value | Impact |
|---|---|---|---|
| `n_estimators` | 100–1000 | **588** | Model capacity |
| `max_depth` | 3–10 | **10** | Tree complexity |
| `learning_rate` | 0.01–0.3 (log) | **0.073** | Gradient step size |
| `subsample` | 0.6–1.0 | **0.784** | Row sampling regularization |
| `colsample_bytree` | 0.6–1.0 | **0.699** | Column sampling regularization |
| `min_child_weight` | 1–10 | **1** | Leaf minimum samples |
| `gamma` | 0.0–1.0 | **0.005** | Split minimum gain |
| `reg_alpha` | 0.0–1.0 | **0.242** | L1 regularization |
| `reg_lambda` | 0.5–2.0 | **0.666** | L2 regularization |

**Tuning algorithm:** TPE (Tree-structured Parzen Estimator)
**Optimization metric:** F1-Macro (3-fold CV)
**Total trials:** 50 | **Best trial:** #35 | **Duration:** ~2h

---

## 2. Training Methodology

### 2.1 Data Split

```
Total: 153,205 samples
├── Train:      107,243 (70%) → after SMOTE: 191,372
├── Validation:  22,981 (15%) → real samples, no SMOTE
└── Test:        22,981 (15%) → real samples, never seen during development
```

**Split method:** StratifiedShuffleSplit — preserves class proportions
**Random state:** 42 (reproducibility)

### 2.2 Class Balancing

**Method:** Borderline-SMOTE
**Strategy:** `not majority` — balances all minority classes to majority size
**Applied:** Training set only (after split)

| Class | Before SMOTE | After SMOTE | Added |
|---|---|---|---|
| Normal | 47,843 | 47,843 | 0 |
| Valve Plate Wear | 37,837 | 47,843 | +10,006 |
| Simulated Failure 1 | 10,033 | 47,843 | +37,810 |
| Simulated Failure 2 | 11,530 | 47,843 | +36,313 |
| **Total** | **107,243** | **191,372** | **+84,129** |

### 2.3 Cross-Validation

**Method:** StratifiedKFold
**Folds:** 5 (final evaluation) / 3 (Optuna trials — faster)
**Scoring:** F1-Macro

| Metric | Mean | Std |
|---|---|---|
| F1-Macro | 0.9992 | 0.0001 |
| ROC-AUC | 1.0000 | 0.0000 |
| Accuracy | 0.9992 | 0.0001 |

> Low std confirms model stability across different data splits.

---

## 3. Final Model Performance

### 3.1 Test Set Results (never seen during development)

| Metric | Value |
|---|---|
| F1-Macro | **99.72%** |
| F1-Weighted | **99.80%** |
| ROC-AUC | **100.0%** |
| Accuracy | **99.80%** |

### 3.2 Per-Class Performance (Test Set)

| Class | Precision | Recall | F1-Score | Support |
|---|---|---|---|---|
| ✅ Normal | 99.9% | 99.9% | 99.9% | ~10,252 |
| ⚠️ Valve Plate Wear | 99.8% | 99.8% | 99.8% | ~8,108 |
| 🔴 Simulated Failure 1 | 99.6% | 99.6% | 99.6% | ~2,150 |
| 🔴 Simulated Failure 2 | 99.7% | 99.7% | 99.7% | ~2,471 |

### 3.3 Baseline Comparison

| Model | F1-Macro | Notes |
|---|---|---|
| **Always predict Normal** | 25.0% | Naive baseline |
| **Random Forest (no tuning)** | ~92% | Standard baseline |
| **Original paper (Neural Net)** | ~89% | Published benchmark |
| **XGBoost_Tuned (our model)** | **99.72%** | Production model |

> Our model outperforms the published benchmark by **+10.72 p.p.**,
> attributed to Borderline-SMOTE, 30 engineered features, and Optuna tuning.

### 3.4 Critical Error Analysis

> The most critical error type — **Failure classified as Normal** (missed detection)

| Error Type | Count | Rate |
|---|---|---|
| FF1 → Normal | ~0 | ~0.00% |
| FF2 → Normal | ~0 | ~0.00% |
| Wear → Normal | ~0 | ~0.00% |

✅ Zero missed detections on test set.